# SMILES → XYZ for small organic molecules

This notebook turns SMILES strings into 3D Cartesian coordinates (`.xyz`) using **RDKit**:

1. Parse the SMILES and add explicit hydrogens.
2. Generate several 3D conformers with the ETKDGv3 distance-geometry method.
3. Optimize each conformer with the MMFF94 force field (UFF as a fallback when MMFF has no parameters).
4. Keep the lowest-energy conformer and write it as an `.xyz` file.

The geometries are force-field quality — a good starting point for DFT/xTB, not a substitute for a QM optimization.

In [ ]:
# Run once if RDKit is not installed (works in Colab too)
# %pip install rdkit

In [ ]:
from pathlib import Path
import re, zipfile
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem.rdForceFieldHelpers import (
    MMFFHasAllMoleculeParams, MMFFOptimizeMoleculeConfs, UFFOptimizeMoleculeConfs)


def safe_name(text):
    """Make a string usable as a file name."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")[:60] or "molecule"


def smiles_to_xyz(smiles, name=None, n_confs=10, seed=42, max_iters=2000):
    """Return (xyz_text, info_dict) for one SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Could not parse SMILES: {smiles!r}")
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params))
    if not conf_ids:  # hard cases: retry with random starting coordinates
        params.useRandomCoords = True
        conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params))
    if not conf_ids:
        raise RuntimeError(f"3D embedding failed for {smiles!r}")

    if MMFFHasAllMoleculeParams(mol):
        results = MMFFOptimizeMoleculeConfs(mol, maxIters=max_iters)
        method = "ETKDGv3 + MMFF94"
    else:
        results = UFFOptimizeMoleculeConfs(mol, maxIters=max_iters)
        method = "ETKDGv3 + UFF"

    # results[i] = (not_converged_flag, energy); pick the lowest energy
    energies = [e for (_, e) in results]
    best = min(range(len(conf_ids)), key=lambda i: energies[i])
    conf = mol.GetConformer(conf_ids[best])

    name = name or safe_name(smiles)
    charge = Chem.GetFormalCharge(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    # XYZ format: line 1 = atom count, line 2 = comment (left blank on purpose)
    lines = [str(mol.GetNumAtoms()), ""]
    for atom in mol.GetAtoms():
        p = conf.GetAtomPosition(atom.GetIdx())
        lines.append(f"{atom.GetSymbol():<2} {p.x:12.6f} {p.y:12.6f} {p.z:12.6f}")
    info = dict(name=name, smiles=smiles, formula=formula, charge=charge,
                n_atoms=mol.GetNumAtoms(), method=method,
                energy_kcal_mol=energies[best],
                converged=results[best][0] == 0, n_conformers=len(conf_ids))
    return "\n".join(lines) + "\n", info

## Convert one molecule

In [ ]:
xyz, info = smiles_to_xyz("CC(=O)Oc1ccccc1C(=O)O", name="aspirin")
print(xyz)
print(info)
Path("aspirin.xyz").write_text(xyz)

## Convert a batch

Edit the dictionary (name → SMILES). Each molecule gets its own `.xyz` file, plus a combined multi-frame file and a zip of everything.

In [ ]:
molecules = {
    "ethanol":   "CCO",
    "benzene":   "c1ccccc1",
    "L-alanine": "C[C@H](N)C(=O)O",
    "caffeine":  "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "ammonium":  "[NH4+]",
}

out_dir = Path("xyz_out")
out_dir.mkdir(exist_ok=True)
summary, combined = [], []

for name, smi in molecules.items():
    try:
        xyz, info = smiles_to_xyz(smi, name=safe_name(name))
        (out_dir / f"{info['name']}.xyz").write_text(xyz)
        combined.append(xyz)
        summary.append(info)
        print(f"OK    {name:12s} {info['formula']:12s} {info['n_atoms']:3d} atoms  {info['method']}")
    except Exception as err:
        print(f"FAIL  {name:12s} {err}")

(out_dir / "all_molecules.xyz").write_text("".join(combined))
with zipfile.ZipFile("xyz_out.zip", "w") as z:
    for f in sorted(out_dir.glob("*.xyz")):
        z.write(f, f.name)
print("\nWrote", len(summary), "molecules to", out_dir.resolve(), "and xyz_out.zip")

## Optional: read SMILES from a file

A `.smi`/`.txt` file with one `SMILES [name]` per line, or a CSV with `smiles` and `name` columns.

In [ ]:
def read_smiles_file(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        import csv
        with open(path, newline="") as fh:
            rows = csv.DictReader(fh)
            return {r.get("name") or safe_name(r["smiles"]): r["smiles"] for r in rows}
    mols = {}
    for line in path.read_text().splitlines():
        parts = line.split()
        if not parts or parts[0].startswith("#"):
            continue
        mols[parts[1] if len(parts) > 1 else safe_name(parts[0])] = parts[0]
    return mols

# molecules = read_smiles_file("my_molecules.smi")   # then re-run the batch cell

## Optional: quick 3D look

Needs `py3Dmol` (`%pip install py3Dmol`).

In [ ]:
try:
    import py3Dmol
    xyz, _ = smiles_to_xyz("Cn1cnc2c1c(=O)n(C)c(=O)n2C", name="caffeine")
    view = py3Dmol.view(width=400, height=300)
    view.addModel(xyz, "xyz")
    view.setStyle({"stick": {}})
    view.zoomTo()
    view.show()
except ImportError:
    print("py3Dmol not installed; skipping the viewer.")

## Notes

- Stereochemistry written in the SMILES (`@`, `@@`, `/`, `\\`) is respected. If it is left out, RDKit picks a stereoisomer for you.
- The lowest of `n_confs` force-field conformers is not guaranteed to be the global minimum; raise `n_confs` for flexible molecules.
- The second line of each `.xyz` file is left blank on purpose (the XYZ format requires the line to exist, but it may be empty). Charge is printed in the `info` summary instead; spin multiplicity is not inferred.
- Metals and unusual bonding often lack MMFF parameters; UFF is used then, and results should be checked visually.